In [31]:
import datasets
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from sklearn.metrics import accuracy_score
import numpy as np
import torchvision
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [32]:
train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomRotation(10),
    torchvision.transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = Imagenette(root = './data', split = 'train', download = True, transform = train_transforms)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = False, num_workers = 3)

In [33]:
validation_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

validation_dataset = Imagenette(root = './data', split = 'val', download = True, transform = validation_transforms)
validation_loader = DataLoader(validation_dataset, batch_size = 32, shuffle = False, num_workers = 3)

Попробуем в лоб запустить модель, обученную на Imagenet1k, на датасете Imagenette, и протестировать её по метрике accuracy.

In [44]:
def AccuracyTest(model, device, data_loader):
  model.eval()
  model.to(device)

  correct = 0
  total = 0

  with torch.no_grad():
      for images, labels in data_loader:
          images = images.to(device)
          labels = labels.to(device)
          
          outputs = model(images)
          _, predicted = torch.max(outputs.data, 1)
          
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  accuracy = correct / total
  # print(accuracy)
  print(f"Accuracy Score: ", accuracy)
  print(f"Correct / Total: {correct} / {total}")

In [45]:
from torchvision.models import resnet50, ResNet50_Weights
print(f"Model Resnet50: ")
AccuracyTest(model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2), device = DEVICE, data_loader = validation_loader)

Model Resnet50: 
Accuracy Score:  0.09579617834394905
Correct / Total: 376 / 3925


Маловато. Попробуем изменить модель: Отрежем её последний слой с 1000 нейронами, а на его место добавим новый слой на 10 нейронов, которые будут обучены на классификацию imagenette.

In [39]:
from torchvision.models import resnet50, ResNet50_Weights

modified_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

for param in modified_model.parameters():
  param.requires_grad = False

last_layer = modified_model.fc.in_features
modified_model.fc = torch.nn.Linear(last_layer, 10)


In [40]:
# Обучение модели на imagenette
import torch.optim as optim
from tqdm import tqdm

epochs_count = 15

optimizer = optim.Adam(modified_model.fc.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()
train_losses = []
validation_accuracies = []

# Лучшие параметры
best_acc = 0
best_model_wts = None

modified_model = modified_model.to(DEVICE)

# Обучение
for epoch in range(1, epochs_count + 1):
  print(f"Epoch: {epoch} / {epochs_count}")
  modified_model.train()

  curr_loss = 0
  curr_corrects = 0

  for inputs, labels in train_loader:
    inputs = inputs.to(DEVICE)
    labels = labels.to(DEVICE)
    optimizer.zero_grad()
    with torch.set_grad_enabled(True):
      outputs = modified_model(inputs)
      _, preds = torch.max(outputs, 1)
      loss = criterion(outputs, labels)

      loss.backward()

      optimizer.step()

    curr_loss += loss.item() * inputs.size(0)
    curr_corrects += torch.sum(preds == labels.data)
  epoch_loss = curr_loss / len(train_loader.dataset)
  epoch_acc = curr_corrects.double() / len(train_loader.dataset)

  train_losses.append(epoch_loss)

  # Валидация
  modified_model.eval()
  val_corrects = 0
  for inputs, labels in validation_loader:
    inputs = inputs.to(DEVICE)
    labels = labels.to(DEVICE)

    with torch.set_grad_enabled(False):
      outputs = modified_model(inputs)
      _, preds = torch.max(outputs, 1)

    val_corrects += torch.sum(preds == labels.data)

  val_acc = val_corrects.double() / len(validation_loader.dataset)
  validation_accuracies.append(val_acc.item())

  print(f"Train Loss: {epoch_loss}, Train Acc: {epoch_acc}")
  print(f"Val Acc: {val_acc}")

  if val_acc > best_acc:
    best_acc = val_acc
    best_model_wts = modified_model.state_dict().copy()

modified_model.load_state_dict(best_model_wts)

Epoch: 1 / 15
Train Loss: 3.00561547181721, Train Acc: 0.48315556024923434
Val Acc: 0.09936305732484076
Epoch: 2 / 15
Train Loss: 2.652712051859461, Train Acc: 0.49160418206780016
Val Acc: 0.09936305732484076
Epoch: 3 / 15
Train Loss: 2.057343168088757, Train Acc: 0.5454641461611575
Val Acc: 0.11210191082802547
Epoch: 4 / 15
Train Loss: 1.8242790265157716, Train Acc: 0.5792586334354208
Val Acc: 0.17452229299363056
Epoch: 5 / 15
Train Loss: 1.7070566390218402, Train Acc: 0.6027035589819411
Val Acc: 0.2578343949044586
Epoch: 6 / 15
Train Loss: 1.6197157789823127, Train Acc: 0.6217129580737142
Val Acc: 0.3498089171974522
Epoch: 7 / 15
Train Loss: 1.5401504185321793, Train Acc: 0.6361812229380083
Val Acc: 0.4114649681528662
Epoch: 8 / 15
Train Loss: 1.4632709874112655, Train Acc: 0.6512831344386947
Val Acc: 0.4552866242038216
Epoch: 9 / 15
Train Loss: 1.424923251873109, Train Acc: 0.6641672827120076
Val Acc: 0.49808917197452224
Epoch: 10 / 15
Train Loss: 1.361554078207959, Train Acc: 0.672

<All keys matched successfully>

In [41]:
torch.save({
    'model_state_dict': modified_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'modified_resnet50.pth')

In [46]:
print("Modified Model Resnet50:")
AccuracyTest(model = modified_model, device = DEVICE, data_loader = validation_loader)


Modified Model Resnet50:
Accuracy Score:  0.5989808917197452
Correct / Total: 2351 / 3925
